## Data Fetch

In [ ]:
from sklearn.datasets import fetch_openml
from sklearn.model_selection import train_test_split
import numpy as np

"""
Open a data set, with name, and load $sample_fraction$ of it.
returns X_train(sampled), Y_train(sampled), X_test, Y_test
dataset names: mnist: mnist_784
"""
def openDataset(name, as_frame = False, sample_fraction = 0.5):

  mnist = fetch_openml(name, as_frame=as_frame)
  X, Y = mnist.data, mnist.target
  X_train, X_test, Y_train, Y_test = train_test_split(
      X, Y, test_size=0.2, random_state=42, stratify=Y)

  # Define the fraction of the data to use (e.g., 50%)
  # Perform stratified sampling to ensure class balance
  sampled_indices = []
  # Convert Y_train to string for consistent type comparison, if needed
  Y_train_str = Y_train.astype(str)

  for class_label in np.unique(Y_train_str):
      class_indices = np.where(Y_train_str == class_label)[0]
      num_samples_in_class = int(len(class_indices) * sample_fraction)

      np.random.seed(42) # for reproducibility per class
      sampled_class_indices = np.random.choice(class_indices, num_samples_in_class, replace=False)
      sampled_indices.extend(sampled_class_indices)

  # Shuffle the final sampled indices to mix the classes
  np.random.shuffle(sampled_indices)

  X_train_sampled = X_train[sampled_indices]
  Y_train_sampled = Y_train[sampled_indices]
  return X_train_sampled, Y_train_sampled, X_test, Y_test


## Ex1

In [ ]:
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import cross_val_score
from sklearn.model_selection import GridSearchCV
from sklearn.ensemble import RandomForestClassifier

model = RandomForestClassifier()

X_train, Y_train, X_test, Y_test = openDataset('mnist_784', sample_fraction=0.2)

# knc = KNeighborsClassifier(n_neighbors=10, weights = 'uniform', algorithm = 'kd_tree')
parameters = {
    'n_estimators': [50, 75, 100],
    'criterion': ['gini', 'entropy'],
    'max_depth': [None, 5, 10],
}
# cross_val_score(knc, X_train, Y_train, cv=3, scoring="accuracy", n_jobs=-1)

scoring = {
    "f1": "f1_macro",
    "precision": "precision_macro",
    "recall": "recall_macro"
}


gscv = GridSearchCV(estimator = model, param_grid = parameters, scoring = scoring,
                    refit = 'f1', cv = 3, n_jobs = -1)
gscv.fit(X_train, Y_train)
print(gscv.best_params_)
print(gscv.best_score_)


{'criterion': 'gini', 'max_depth': None, 'n_estimators': 100}
0.9416591636323033


In [ ]:
from sklearn.metrics import f1_score
from sklearn.metrics import accuracy_score

best_model = gscv.best_estimator_
y_pred = best_model.predict(X_test)

f1 = f1_score(Y_test, y_pred, average="macro")
acc = accuracy_score(Y_test, y_pred)
print(f"Test F1: {f1:.4f}")
print(f"Accuracy: {acc:.4f}")

Test F1: 0.9480
Accuracy: 0.9485


## Ex 2

In [ ]:
from IPython.core.display import Image
X_train, Y_train, X_test, Y_test = openDataset('mnist_784', sample_fraction=0.2)

print(X_train.shape)
def shift_x(image, n):
    if n == 0: return image
    shape = list(image.shape)
    if n>=0:

      shifted_image = image[:, :-n] if n < len(image)  else np.array([], dtype=image.dtype)
      pad_shape = ( *shape[:1], n)
      padding = np.zeros(pad_shape, dtype=image.dtype)
      return np.concatenate([padding, shifted_image], axis=1)
    else :
      n = -n
      shifted_image = image[:, n:] if n < len(image)  else np.array([], dtype=image.dtype)
      pad_shape = (*shape[:1], n)
      padding = np.zeros(pad_shape, dtype=image.dtype)
      return np.concatenate([shifted_image, padding], axis=1)


example = np.array([[1,2,3,4], [5,6,7,8],[9,10,11,12]])
def shift_y(image, n):
  if n == 0:
    return image
  shape = list(image.shape)
  if n>=0:
    shifted_image = image[:-n, :] if n < len(image)  else np.array([], dtype=image.dtype)
    pad_shape = (n, *shape[1:])
    padding = np.zeros(pad_shape, dtype=image.dtype)
    return np.concatenate([padding, shifted_image], axis=0)
  else :
    n = -n
    shifted_image = image[n:, :] if n < len(image)  else np.array([], dtype=image.dtype)
    pad_shape = (n, *shape[1:])
    padding = np.zeros(pad_shape, dtype=image.dtype)
    return np.concatenate([shifted_image, padding], axis=0)


X_train = X_train.reshape((-1, 28, 28))
print(X_train.shape)

X_augmented = []
Y_augmented = []

for x, y in zip(X_train, Y_train):
  X_augmented.append(x)
  Y_augmented.append(y)

  X_augmented.append(shift_x(x, 1))
  Y_augmented.append(y)

  X_augmented.append(shift_x(x, -1))
  Y_augmented.append(y)

  X_augmented.append(shift_y(x, 1))
  Y_augmented.append(y)

  X_augmented.append(shift_y(x, -1))
  Y_augmented.append(y)\

X_augmented = np.array(X_augmented)
Y_augmented = np.array(Y_augmented)

print(X_augmented.shape)


(11196, 784)
(11196, 28, 28)
(55980, 28, 28)


In [ ]:
from sklearn.metrics import f1_score
from sklearn.metrics import accuracy_score

best_model = gscv.best_estimator_
best_model.fit(X_augmented.reshape(-1, 784), Y_augmented)
y_pred = best_model.predict(X_test)

f1 = f1_score(Y_test, y_pred, average="macro")
acc = accuracy_score(Y_test, y_pred)
print(Y_test.shape)
print(f"Test F1: {f1:.4f}")
print(f"Accuracy: {acc:.4f}")

(14000,)
Test F1: 0.9654
Accuracy: 0.9656
